# Ingest: Medicare Inpatient Claims

Healthcare demo data. Downloads CMS's synthetic DE-SynPUF inpatient
claims sample (2008-2010) and persists it as a managed Spark table — one
row per inpatient claim, joinable to `medicare_beneficiary_ingest.ipynb`
on `DESYNPUF_ID`.

Companion to `medicare_beneficiary_ingest.ipynb` — both feed
`data-transformation/readmission_risk_summary.ipynb`.

Re-runnable: uses `CREATE DATABASE IF NOT EXISTS` and
`mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "medicare_claims"
TABLE_NAME = "inpatient_claims"
SOURCE_URL = (
    "https://www.cms.gov/research-statistics-data-and-systems/downloadable-public-use-files/"
    "synpufs/downloads/de1_0_2008_to_2010_inpatient_claims_sample_1.zip"
)
LOCAL_ZIP = "/tmp/de1_0_2008_to_2010_inpatient_claims_sample_1.zip"
LOCAL_DIR = "/tmp/medicare_inpatient"

## Download source file

A `User-Agent` header is required — CMS's server blocks requests without
one (confirmed live, 403).

In [ ]:
import urllib.request

request = urllib.request.Request(SOURCE_URL, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(request) as response, open(LOCAL_ZIP, "wb") as out_file:
    out_file.write(response.read())

## Extract CSV

In [ ]:
import glob
import zipfile

with zipfile.ZipFile(LOCAL_ZIP) as zf:
    zf.extractall(LOCAL_DIR)

csv_files = glob.glob(f"{LOCAL_DIR}/*.csv")
csv_files

## Load dataset

Key columns used downstream: `DESYNPUF_ID` (beneficiary), `CLM_ADMSN_DT`
(admission date), `NCH_BENE_DSCHRG_DT` (discharge date), `ICD9_DGNS_CD_1`
(primary diagnosis), `CLM_PMT_AMT` (claim payment amount).

In [ ]:
df = spark.read.csv(csv_files, header=True, inferSchema=True)

## Create schema/database

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
COMMENT 'Synthetic Medicare claims data (CMS DE-SynPUF) used for healthcare demonstrations'
""")

## Persist as a managed Spark table

In [ ]:
df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TABLE_NAME}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_claims FROM {DATABASE_NAME}.{TABLE_NAME}").show()
spark.sql(f"DESCRIBE {DATABASE_NAME}.{TABLE_NAME}").show(truncate=False)